In [0]:
results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True, inferSchema=True
).toPandas()
print(f"Results shape: {results.shape}")
results.head()


In [0]:
races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True, inferSchema=True
).toPandas()
print(f"Races shape: {races.shape}")
races.head()

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Merge results with race metadata
df = results.merge(races[['raceId', 'year', 'round']], on='raceId', how='left')

keep_cols = ['raceId', 'driverId', 'constructorId', 'grid', 'laps',
             'year', 'round', 'fastestLapSpeed', 'positionOrder']
df = df[keep_cols].copy()

df['fastestLapSpeed'] = pd.to_numeric(df['fastestLapSpeed'], errors='coerce')
df = df.dropna().reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")

feature_cols = ['driverId', 'constructorId', 'grid', 'laps',
                'year', 'round', 'fastestLapSpeed']
target_col = 'positionOrder'

X = df[feature_cols]
y = df[target_col]
race_ids = df['raceId']

X_train, X_test, y_train, y_test, race_train, race_test = train_test_split(
    X, y, race_ids, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
df.head()

In [0]:
# ============================================================
# Q1: Create two tables to store predictions from each model
# ============================================================

CATALOG  = "workspace"
SCHEMA   = "default"
TABLE_RF = f"{CATALOG}.{SCHEMA}.hw5_helle_predictions_rf"
TABLE_GB = f"{CATALOG}.{SCHEMA}.hw5_helle_predictions_gb"

# Drop tables completely
spark.sql(f"DROP TABLE IF EXISTS {TABLE_RF}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_GB}")

# Create empty tables by writing an empty Spark DataFrame
# This way the schema is determined by Spark itself (no type mismatch)
from pyspark.sql.types import (StructType, StructField,
                                LongType, DoubleType, StringType)

schema = StructType([
    StructField("race_id",            LongType(),   True),
    StructField("driver_id",          LongType(),   True),
    StructField("constructor_id",     LongType(),   True),
    StructField("grid",               LongType(),   True),
    StructField("actual_position",    LongType(),   True),
    StructField("predicted_position", DoubleType(), True),
    StructField("model_run_id",       StringType(), True),
])

empty_df = spark.createDataFrame([], schema)
empty_df.write.mode("overwrite").saveAsTable(TABLE_RF)
empty_df.write.mode("overwrite").saveAsTable(TABLE_GB)

print(f"Created table: {TABLE_RF}")
print(f"Created table: {TABLE_GB}")
spark.sql(f"DESCRIBE {TABLE_RF}").show()
spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set MLflow experiment
mlflow.set_experiment("/Users/yh3972@columbia.edu/hw5_f1_predictions")


def save_residuals_plot(y_test, y_pred, filepath="/tmp/residuals.png"):
    """Artifact 1: residuals scatter plot."""
    residuals = np.array(y_test) - np.array(y_pred)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(y_pred, residuals, alpha=0.4, color='steelblue')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel("Predicted Values")
    ax.set_ylabel("Residuals")
    ax.set_title("Residuals Plot")
    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()
    return filepath


def save_feature_importance_csv(model, feature_names, filepath="/tmp/feature_importance.csv"):
    """Artifact 2: feature importance CSV."""
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    importance_df.to_csv(filepath, index=False)
    return filepath


print("MLflow and helper functions ready!")

In [0]:
def train_and_log_model(model, model_name, run_name, table_name, params):
    """
    Train a model, log everything to MLflow, and write predictions to DB.
    """
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        # --- Train ---
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # --- Log hyperparameters ---
        for param_name, param_value in params.items():
            mlflow.log_param(param_name, param_value)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("test_size",  0.2)

        # --- Log 4 metrics ---
        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        mlflow.log_metric("mae",  mae)
        mlflow.log_metric("mse",  mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2",   r2)

        # --- Log model ---
        mlflow.sklearn.log_model(model, artifact_path=model_name)

        # --- Log 2 artifacts ---
        mlflow.log_artifact(
            save_residuals_plot(y_test, y_pred),
            artifact_path="plots"
        )
        mlflow.log_artifact(
            save_feature_importance_csv(model, X_train.columns.tolist()),
            artifact_path="plots"
        )

        # --- Write predictions to database ---
        predictions_df = pd.DataFrame({
            "race_id":            race_test.values.astype(int),
            "driver_id":          X_test["driverId"].values.astype(int),
            "constructor_id":     X_test["constructorId"].values.astype(int),
            "grid":               X_test["grid"].values.astype(int),
            "actual_position":    y_test.values.astype(int),
            "predicted_position": y_pred.astype(float),
            "model_run_id":       run_id
        })

        spark_df = spark.createDataFrame(predictions_df)
        spark_df.write.mode("append").saveAsTable(table_name)

        print(f"[{run_name}] MAE={mae:.3f} | RMSE={rmse:.3f} | R2={r2:.4f}")
        print(f"  Wrote {len(predictions_df)} predictions to {table_name}")
        print(f"  MLflow run_id: {run_id}")

        return {"run_name": run_name, "run_id": run_id,
                "mae": mae, "mse": mse, "rmse": rmse, "r2": r2}


print("train_and_log_model function ready!")



In [0]:
# ============================================================
# Q2/Q3 Model 1: Random Forest Regressor
# ============================================================

rf_params = {
    "n_estimators":      300,
    "max_depth":         15,
    "min_samples_split": 2,
    "min_samples_leaf":  1,
    "random_state":      42
}

rf_model = RandomForestRegressor(**rf_params, n_jobs=-1)

rf_result = train_and_log_model(
    model      = rf_model,
    model_name = "random_forest_regressor",
    run_name   = "RandomForest_n300_depth15",
    table_name = TABLE_RF,
    params     = rf_params
)

In [0]:
# ============================================================
# Q2/Q3 Model 2: Gradient Boosting Regressor
# ============================================================

gb_params = {
    "n_estimators":  300,
    "max_depth":     5,
    "learning_rate": 0.05,
    "subsample":     0.8,
    "random_state":  42
}

gb_model = GradientBoostingRegressor(**gb_params)

gb_result = train_and_log_model(
    model      = gb_model,
    model_name = "gradient_boosting_regressor",
    run_name   = "GradientBoosting_n300_lr0.05",
    table_name = TABLE_GB,
    params     = gb_params
)

In [0]:
# ============================================================
# Verify predictions were written to both tables
# ============================================================

print(f"=== {TABLE_RF} ===")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_RF}").show()
spark.sql(f"SELECT * FROM {TABLE_RF} LIMIT 5").show()

print(f"\n=== {TABLE_GB} ===")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE_GB}").show()
spark.sql(f"SELECT * FROM {TABLE_GB} LIMIT 5").show()